# NLP Week 1 Practice Review  
## Regular Expressions, Text Cleaning, and Tokenization

This notebook is a personal review artifact for Week 1 of a Natural Language Processing course within a Master's program in Applied Artificial Intelligence.

The purpose is to practice how regular expressions help us identify, clean, and transform patterns in raw text before applying more advanced NLP techniques.

**Core topics covered**

1. Loading a text corpus.
2. Removing sentiment labels and line breaks.
3. Detecting linguistic and symbolic patterns with regex.
4. Cleaning raw text into a normalized corpus.
5. Tokenizing text into words.
6. Building a vocabulary.
7. Removing stopwords to reduce noise.

> This version is intentionally written as a public-facing study guide. It avoids institutional references and focuses on reusable NLP concepts.

## Dataset context

The examples are designed around the `yelp_labelled.txt` file from the Sentiment Labelled Sentences dataset. Each row contains a short Yelp review followed by a sentiment label:

```text
Wow... Loved this place.\t1\n
Crust is not good.\t0\n
```

For this Week 1 review, the labels are useful only as metadata. The main goal is to practice text cleaning and pattern extraction, so the first step is to remove the labels and keep only the review text.

## 1. Imports and data loading

Only two libraries are required for this exercise:

- `re`: Python's regular expression module.
- `numpy`: included for consistency with basic data-handling exercises, although most of the work here is text-oriented.

The code below expects `yelp_labelled.txt` to be located in the same folder as this notebook. If the file is not found, a small sample corpus is used so the notebook can still demonstrate the logic.

In [1]:
import re
from pathlib import Path
import numpy as np

DATA_FILE = Path("yelp_labelled.txt")

sample_docs = [
    "Wow... Loved this place.\t1\n",
    "Crust is not good.\t0\n",
    "The food was AMAZING!!!\t1\n",
    "I will NEVER go back.\t0\n",
    "The bill was $20.50 and the service was sooooo slow.\t0\n",
    "Hands-down one of the best places I have visited.\t1\n",
    "DELICIOUS!!\t1\n",
    "The experience was disappointing and overpriced.\t0\n",
]

if DATA_FILE.exists():
    with DATA_FILE.open("r", encoding="utf-8") as file:
        docs = file.readlines()
    print(f"Loaded {len(docs)} rows from {DATA_FILE}.")
else:
    docs = sample_docs
    print("Dataset file not found. Using a small sample corpus for demonstration.")

print(type(docs))
print(docs[:3])

Dataset file not found. Using a small sample corpus for demonstration.
<class 'list'>
['Wow... Loved this place.\t1\n', 'Crust is not good.\t0\n', 'The food was AMAZING!!!\t1\n']


## 2. Removing labels and line breaks

The original corpus follows this structure:

```text
review text\tlabel\n
```

The pattern below removes the tab character, the binary sentiment label, and the trailing line break.

```python
r"\t[01]\s*$"
```

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\t` | Matches the tab that separates the review from its label. |
| `[01]` | Matches either `0` or `1`, the sentiment label. |
| `\s*` | Matches any optional whitespace after the label. |
| `$` | Anchors the match to the end of the string. |

This is a common preprocessing step in NLP: remove target labels or metadata before analyzing the text itself.

In [2]:
label_pattern = r"\t[01]\s*$"

clean_reviews = [re.sub(label_pattern, "", review) for review in docs]

for i, review in enumerate(clean_reviews[:10]):
    print(f"{i}: {review}")

0: Wow... Loved this place.
1: Crust is not good.
2: The food was AMAZING!!!
3: I will NEVER go back.
4: The bill was $20.50 and the service was sooooo slow.
5: Hands-down one of the best places I have visited.
6: DELICIOUS!!
7: The experience was disappointing and overpriced.


## 3. Helper function for repeated regex searches

Because several tasks follow the same structure - apply a pattern to every document, collect matches, and count them - it is useful to define a small reusable helper function.

In [3]:
def find_matches(corpus, pattern, flags=0):
    """
    Apply a regex pattern to every document in a corpus and return all matches.
    """
    matches = []
    for text in corpus:
        matches.extend(re.findall(pattern, text, flags=flags))
    return matches


def print_summary(matches, title="Matches", max_items=30):
    """
    Print a compact preview of regex results.
    """
    print(title)
    print("-" * len(title))
    for item in matches[:max_items]:
        print(item)
    if len(matches) > max_items:
        print(f"... {len(matches) - max_items} additional matches not shown")
    print(f"\nTotal matches: {len(matches)}")

## 4. Words followed by two or more exclamation marks

```python
r"\b[A-Za-z]+!{2,}"
```

**What it captures**

Words such as `amazing!!!`, `DELICIOUS!!`, or `Firehouse!!!!!`.

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\b` | Word boundary. It helps start the match at the beginning of a word. |
| `[A-Za-z]+` | One or more alphabetic characters. |
| `!{2,}` | Two or more exclamation marks. |

**Why it matters in NLP**

Repeated punctuation often signals emphasis, intensity, frustration, excitement, or strong sentiment. These signals may be relevant in sentiment analysis and customer review mining.

In [4]:
pattern_exclamations = r"\b[A-Za-z]+!{2,}"
exclamation_matches = find_matches(clean_reviews, pattern_exclamations)
print_summary(exclamation_matches, "Words followed by repeated exclamation marks")

Words followed by repeated exclamation marks
--------------------------------------------
AMAZING!!!
DELICIOUS!!

Total matches: 2


## 5. Fully uppercase words with two or more letters

```python
r"\b[A-Z]{2,}\b"
```

**What it captures**

Words such as `NEVER`, `DELICIOUS`, `WORST`, or `OMG`.

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\b` | Word boundary. |
| `[A-Z]` | Any uppercase English letter. |
| `{2,}` | At least two consecutive uppercase letters. |
| `\b` | Closing word boundary. |

**Why it matters in NLP**

Uppercase words can act as pragmatic markers. In user-generated content, they often reflect emphasis, shouting, frustration, excitement, or brand-like expressions.

In [5]:
pattern_uppercase_words = r"\b[A-Z]{2,}\b"
uppercase_words = find_matches(clean_reviews, pattern_uppercase_words)
print_summary(uppercase_words, "Uppercase words")

Uppercase words
---------------
AMAZING
NEVER
DELICIOUS

Total matches: 3


## 6. Comments where all alphabetic characters are uppercase

```python
r"^(?=.*[A-Za-z])[^a-z]+$"
```

**What it captures**

Full comments such as `DELICIOUS!!` or `TOTAL WASTE OF TIME.` where every alphabetic character is uppercase.

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `^` | Start of the string. |
| `(?=.*[A-Za-z])` | Positive lookahead requiring at least one alphabetic character. |
| `[^a-z]+` | One or more characters that are not lowercase letters. |
| `$` | End of the string. |

**Why it matters in NLP**

A fully uppercase sentence can behave differently from normal casing. It may indicate emotional intensity or strong polarity.

In [6]:
pattern_all_caps_comment = r"^(?=.*[A-Za-z])[^a-z]+$"
all_caps_comments = [text for text in clean_reviews if re.search(pattern_all_caps_comment, text)]
print_summary(all_caps_comments, "Comments with all alphabetic characters in uppercase")

Comments with all alphabetic characters in uppercase
----------------------------------------------------
DELICIOUS!!

Total matches: 1


## 7. Words containing accented vowels

```python
r"\b[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]*[ÁÉÍÓÚáéíóú][A-Za-zÁÉÍÓÚÜÑáéíóúüñ]*\b"
```

**What it captures**

Words containing accented vowels such as `café`, `jalapeño`, or `menú` if they appear in the corpus.

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\b` | Word boundary. |
| `[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]*` | Optional alphabetic characters, including Spanish letters. |
| `[ÁÉÍÓÚáéíóú]` | Requires at least one accented vowel. |
| `[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]*` | Optional alphabetic characters after the accented vowel. |
| `\b` | Closing word boundary. |

**Why it matters in NLP**

Accent handling is important in multilingual text. If accents are removed incorrectly, different words may collapse into the same token and lose meaning.

In [7]:
pattern_accented_words = r"\b[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]*[ÁÉÍÓÚáéíóú][A-Za-zÁÉÍÓÚÜÑáéíóúüñ]*\b"
accented_words = find_matches(clean_reviews, pattern_accented_words)
print_summary(accented_words, "Words containing accented vowels")

Words containing accented vowels
--------------------------------

Total matches: 0


## 8. Monetary amounts starting with `$`

```python
r"\$\s*\d{1,3}(?:,\d{3})*(?:\.\d+)?"
```

**What it captures**

Amounts such as `$20`, `$4.00`, `$1,250`, or `$11.99`.

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\$` | Escapes the dollar sign so it is treated as a literal character. |
| `\s*` | Allows optional whitespace after `$`. |
| `\d{1,3}` | Matches one to three digits before any thousands separator. |
| `(?:,\d{3})*` | Optional groups of comma plus exactly three digits. |
| `(?:\.\d+)?` | Optional decimal part. |

**Why it matters in NLP**

Prices and monetary amounts can be important entities in reviews, complaints, invoices, financial text, and market analysis.

In [8]:
pattern_money = r"\$\s*\d{1,3}(?:,\d{3})*(?:\.\d+)?"
money_amounts = find_matches(clean_reviews, pattern_money)
print_summary(money_amounts, "Monetary amounts")

Monetary amounts
----------------
$20.50

Total matches: 1


## 9. Variants of the word `love`

```python
r"\bl+o+v+e+d?\b"
```

**What it captures**

Variants such as `love`, `Love`, `LOVED`, `loved`, or exaggerated forms like `looove`.

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\b` | Word boundary. |
| `l+` | One or more `l` characters. |
| `o+` | One or more `o` characters. |
| `v+` | One or more `v` characters. |
| `e+` | One or more `e` characters. |
| `d?` | Optional `d`, allowing `love` and `loved`. |
| `\b` | Closing word boundary. |

**Why it matters in NLP**

Informal text often contains elongated spellings. Regex can help capture expressive variants without depending on exact spelling only.

In [9]:
pattern_love = r"\bl+o+v+e+d?\b"
love_variants = find_matches(clean_reviews, pattern_love, flags=re.IGNORECASE)
print_summary(love_variants, "Variants of the word 'love'")

Variants of the word 'love'
---------------------------
Loved

Total matches: 1


## 10. Variants of `so` and `good` with repeated `o`

```python
r"\b(?:so{2,}|go{3,}d)\b"
```

**What it captures**

- `soo`, `sooo`, `sooooo`
- `goood`, `gooood`, `goooood`

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\b` | Word boundary. |
| `(?: ... )` | Non-capturing group used for alternatives. |
| `so{2,}` | Letter `s` followed by two or more `o` characters. |
| `|` | Logical OR. |
| `go{3,}d` | Letter `g`, three or more `o` characters, and `d`. |
| `\b` | Closing word boundary. |

**Why it matters in NLP**

Character repetition often encodes intensity. This is especially common in reviews, chats, tweets, and social media comments.

In [10]:
pattern_repeated_o = r"\b(?:so{2,}|go{3,}d)\b"
repeated_o_words = find_matches(clean_reviews, pattern_repeated_o, flags=re.IGNORECASE)
print_summary(repeated_o_words, "Variants of 'so' and 'good' with repeated 'o'")

Variants of 'so' and 'good' with repeated 'o'
---------------------------------------------
sooooo

Total matches: 1


## 11. Alphabetic words longer than 12 characters

```python
r"\b[A-Za-z]{13,}\b"
```

**What it captures**

Words such as `recommendation`, `disappointing`, `underwhelming`, or `unprofessional`.

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\b` | Word boundary. |
| `[A-Za-z]` | Any alphabetic English character. |
| `{13,}` | Thirteen or more characters. |
| `\b` | Closing word boundary. |

**Why it matters in NLP**

Long words may reveal domain-specific vocabulary, misspellings, compound expressions, or stylistic patterns.

In [11]:
pattern_long_words = r"\b[A-Za-z]{13,}\b"
long_words = find_matches(clean_reviews, pattern_long_words)
print_summary(long_words, "Words longer than 12 alphabetic characters")

Words longer than 12 alphabetic characters
------------------------------------------
disappointing

Total matches: 1


## 12. Capitalized words that are not the first word in the comment

```python
r"\b[A-Z][a-z]+\b"
```

This pattern detects words that begin with an uppercase letter and end with lowercase letters. However, the exercise adds one extra condition: the match should not be the first word of the comment.

For that reason, the regex is combined with positional logic using `re.finditer()`.

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\b` | Word boundary. |
| `[A-Z]` | One uppercase initial letter. |
| `[a-z]+` | One or more lowercase letters. |
| `\b` | Closing word boundary. |

**Why it matters in NLP**

Capitalized words inside a sentence often correspond to names, locations, brands, products, or proper nouns. These are useful signals for named entity recognition.

In [12]:
pattern_capitalized = r"\b[A-Z][a-z]+\b"
capitalized_not_first = []

for review in clean_reviews:
    first_word = re.search(r"\b[A-Za-z]+\b", review)
    for match in re.finditer(pattern_capitalized, review):
        if first_word and match.start() == first_word.start():
            continue
        capitalized_not_first.append(match.group())

print_summary(capitalized_not_first, "Capitalized words that are not the first word")

Capitalized words that are not the first word
---------------------------------------------
Loved

Total matches: 1


## 13. Hyphenated word sequences without spaces

```python
r"\b[A-Za-z]+(?:-[A-Za-z]+)+\b"
```

**What it captures**

Expressions such as `hands-down`, `must-stop`, `sub-par`, or `in-house`.

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\b` | Word boundary. |
| `[A-Za-z]+` | First alphabetic word. |
| `(?:-[A-Za-z]+)+` | One or more hyphen-plus-word sequences. |
| `\b` | Closing word boundary. |

**Why it matters in NLP**

Hyphenated expressions can behave like single semantic units. Depending on the task, we may want to preserve them instead of splitting them into separate words.

In [13]:
pattern_hyphenated = r"\b[A-Za-z]+(?:-[A-Za-z]+)+\b"
hyphenated_words = find_matches(clean_reviews, pattern_hyphenated)
print_summary(hyphenated_words, "Hyphenated word sequences")

Hyphenated word sequences
-------------------------
Hands-down

Total matches: 1


## 14. Words ending in `ing` or `ed`

```python
r"\b[A-Za-z]+ing\b"
r"\b[A-Za-z]+ed\b"
```

**What it captures**

- `ing`: `getting`, `amazing`, `running`, `disappointing`
- `ed`: `loved`, `stopped`, `visited`, `overpriced`

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `\b` | Word boundary. |
| `[A-Za-z]+` | One or more alphabetic characters. |
| `ing` or `ed` | Required ending. |
| `\b` | Closing word boundary. |

**Why it matters in NLP**

Suffixes can indicate tense, aspect, or grammatical form. Regex is not a substitute for lemmatization, but it is useful for quick morphological exploration.

In [14]:
pattern_ing = r"\b[A-Za-z]+ing\b"
pattern_ed = r"\b[A-Za-z]+ed\b"

words_ing = find_matches(clean_reviews, pattern_ing, flags=re.IGNORECASE)
words_ed = find_matches(clean_reviews, pattern_ed, flags=re.IGNORECASE)

print_summary(words_ing, "Words ending in 'ing'")
print("\n")
print_summary(words_ed, "Words ending in 'ed'")

Words ending in 'ing'
---------------------
AMAZING
disappointing

Total matches: 2


Words ending in 'ed'
--------------------
Loved
visited
overpriced

Total matches: 3


## 15. Corpus cleaning for tokenization

The next step is to create a cleaner version of the corpus. The cleaning pipeline applies three transformations:

1. Keep alphabetic characters and whitespace only.
2. Convert all letters to lowercase.
3. Collapse repeated whitespace into single spaces.

```python
r"[^A-Za-zÁÉÍÓÚÜÑáéíóúüñ\s]"
```

**Pattern breakdown**

| Component | Meaning |
|---|---|
| `[^ ... ]` | Negated character class. It matches anything not listed inside. |
| `A-Za-z` | English alphabetic characters. |
| `ÁÉÍÓÚÜÑáéíóúüñ` | Spanish alphabetic characters and accents. |
| `\s` | Whitespace. |

Replacing this pattern with a blank space removes punctuation, numbers, symbols, and special characters while preserving the alphabetic content.

In [15]:
cleaning_pattern = r"[^A-Za-zÁÉÍÓÚÜÑáéíóúüñ\s]"
whitespace_pattern = r"\s+"

normalized_reviews = []
for review in clean_reviews:
    text = re.sub(cleaning_pattern, " ", review)
    text = text.lower()
    text = re.sub(whitespace_pattern, " ", text).strip()
    normalized_reviews.append(text)

for i, review in enumerate(normalized_reviews[:10]):
    print(f"{i}: {review}")

0: wow loved this place
1: crust is not good
2: the food was amazing
3: i will never go back
4: the bill was and the service was sooooo slow
5: hands down one of the best places i have visited
6: delicious
7: the experience was disappointing and overpriced


## 16. Tokenization and vocabulary creation

For this introductory exercise, tokenization is performed using whitespace splitting.

```python
review.split()
```

This creates one token per whitespace-separated word.

A vocabulary is then created by converting the full token list into a set.

**Important note**

Whitespace tokenization is simple and useful for a first pass, but it is not always enough. More advanced NLP pipelines may use language-specific tokenizers, subword tokenizers, or model-specific tokenization strategies.

In [16]:
tokens = []
for review in normalized_reviews:
    tokens.extend(review.split())

vocabulary = set(tokens)

print(f"Total tokens: {len(tokens)}")
print(f"Unique tokens in the vocabulary: {len(vocabulary)}")
print(sorted(list(vocabulary))[:30])

Total tokens: 43
Unique tokens in the vocabulary: 34
['amazing', 'and', 'back', 'best', 'bill', 'crust', 'delicious', 'disappointing', 'down', 'experience', 'food', 'go', 'good', 'hands', 'have', 'i', 'is', 'loved', 'never', 'not', 'of', 'one', 'overpriced', 'place', 'places', 'service', 'slow', 'sooooo', 'the', 'this']


## 17. Stopword removal

Stopwords are high-frequency words that often carry limited semantic value in basic bag-of-words workflows. Examples include articles, pronouns, auxiliary verbs, prepositions, and connectors.

Removing them can reduce noise and make the vocabulary more focused on content-bearing words.

However, stopword removal should be used carefully. In some tasks, words such as `not`, `no`, or `never` are highly relevant, especially for sentiment analysis.

In [17]:
stopwords = [
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves',
    'you', 'your', 'yours', 'he', 'him', 'his', 'himself', 'she',
    'her', 'hers', 'herself', 'it', 'its', 'itself', 'they', 'them',
    'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom',
    'this', 'that', 'these', 'those', 'am', 'is', 'are', 'was',
    'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having',
    'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but',
    'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by',
    'for', 'with', 'about', 'against', 'between', 'into', 'through',
    'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up',
    'down', 'in', 'out', 'on', 'over', 'under', 'again', 'further',
    'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how',
    'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other',
    'some', 'such', 'only', 'own', 'same', 'so', 'than', 'too',
    'very', 's', 't', 'can', 'will', 'just', 'should', 'now', 'll'
]

stopword_set = set(stopwords)

reviews_without_stopwords = []
for review in normalized_reviews:
    filtered_tokens = [token for token in review.split() if token not in stopword_set]
    reviews_without_stopwords.append(" ".join(filtered_tokens))

final_tokens = []
for review in reviews_without_stopwords:
    final_tokens.extend(review.split())

final_vocabulary = set(final_tokens)

for i, review in enumerate(reviews_without_stopwords[:5]):
    print(f"{i}: {review}")

print(f"\nFinal token count: {len(final_tokens)}")
print(f"Final unique token count: {len(final_vocabulary)}")

0: wow loved place
1: crust not good
2: food amazing
3: never go back
4: bill service sooooo slow

Final token count: 24
Final unique token count: 24


## 18. Key takeaways

This exercise shows how regular expressions can be used as a practical first layer of text preprocessing.

**Main lessons**

- Regex is effective for extracting explicit surface patterns such as punctuation, casing, prices, suffixes, and hyphenated expressions.
- Text cleaning decisions affect the final vocabulary and the quality of downstream NLP analysis.
- Tokenization defines what counts as a unit of analysis.
- Stopword removal can reduce noise, but it must be aligned with the objective of the analysis.
- Regex is powerful, but it should be complemented with linguistic tools when the task requires deeper semantic understanding.

**Practical conclusion**

Before applying advanced NLP models, it is essential to understand the structure, noise, and linguistic signals present in the raw text. Regex-based exploration is one of the fastest ways to build that understanding.

## 19. Compact regex reference

| Goal | Pattern |
|---|---|
| Remove tab, label, and trailing whitespace | `r"\t[01]\s*$"` |
| Words followed by repeated exclamation marks | `r"\b[A-Za-z]+!{2,}"` |
| Uppercase words | `r"\b[A-Z]{2,}\b"` |
| Comments with no lowercase letters | `r"^(?=.*[A-Za-z])[^a-z]+$"` |
| Words with accented vowels | `r"\b[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]*[ÁÉÍÓÚáéíóú][A-Za-zÁÉÍÓÚÜÑáéíóúüñ]*\b"` |
| Monetary amounts | `r"\$\s*\d{1,3}(?:,\d{3})*(?:\.\d+)?"` |
| Variants of `love` | `r"\bl+o+v+e+d?\b"` |
| Variants of `so` and `good` | `r"\b(?:so{2,}|go{3,}d)\b"` |
| Words longer than 12 letters | `r"\b[A-Za-z]{13,}\b"` |
| Capitalized words | `r"\b[A-Z][a-z]+\b"` |
| Hyphenated words | `r"\b[A-Za-z]+(?:-[A-Za-z]+)+\b"` |
| Words ending in `ing` | `r"\b[A-Za-z]+ing\b"` |
| Words ending in `ed` | `r"\b[A-Za-z]+ed\b"` |
| Remove non-alphabetic characters | `r"[^A-Za-zÁÉÍÓÚÜÑáéíóúüñ\s]"` |
| Collapse extra whitespace | `r"\s+"` |